### Montagem de um Data Lake com DuckDB

Nesse estudo de caso vou usar o Dataset athelic que contem dados de jogos olimpicos do anos de 1984 a 2016
O objectivo e criar a estrutura medalhao usando a biblioteca Duckdb e visualizacao e tratamento de dados usando
o software POWER BI e EXCEL.

In [ ]:
#---------------------------------------------------------------------------------------------------------------------
# Bibliotecas Carregadas no projecto
# os - vamos utilizar a biblioteca os para gestao de caminhos de pastas bem com criar e validacao de pasta e ficheiro
# duckdb - vamos utlizar duckdb para armazenamento e processamento dos dados 
# glob - utilizaremos o glob para leitura de ficheiro de uma pasta (faz varedura na pasta pegando ficheiro .csv)
#---------------------------------------------------------------------------------------------------------------------

import os
import duckdb
from glob import glob

##### Estrutura de Pasta do projecto
- Pasta Bronze vai conter os dados brutos os ficheiros csv
- Pasta Silver vai conter os dados ja processados no banco Duckdb oriundos da camada bronze
- Pasta Gold vai conter os dados ja no formato parquet particionados por anos


In [ ]:
# criacao de estrutura de pasta de trabalho

# criar endereco bronze, silver e gold dentro da pasta data_lake
base_path = "data_lake"
bronze_path = os.path.join(base_path, "bronze")
silver_path = os.path.join(base_path, "silver")
gold_path   = os.path.join(base_path, "gold")

# endereco onde sera criado banco de dados lakehouse
db_path = "duckdb/lakehouse.duckdb"

# criacao das pastas com verificacao de existencia delas
os.makedirs(bronze_path, exist_ok=True)
os.makedirs(silver_path, exist_ok=True)
os.makedirs(gold_path, exist_ok=True)
os.makedirs("duckdb", exist_ok=True)

In [ ]:
#--------------------------------------------------------------------------------------------
# Criamos um banco de dados no disco, ou seja os dados sao persistidos, no disco local
# O banco de dados foi denominado de lakehouse e será criado caso nao existir, proprio duckdb 
# vai gerir isso
#--------------------------------------------------------------------------------------------

con = duckdb.connect(db_path)

##### Vamos criar algumas tabelas uteis Para controle e gestao dos dados 

- Tabela de controle de Arquivos processados - Aqui registramos os ficheiros ja processados e inseridos dados no banco
- Tabela de controle de marcas de agua (watermark) - Registro de data, indice da ultima carga na tabela silver
- Tabela de registro de logs - registro de status de carga falha ou sucesso, data da carga
- Tabela Silver clean - coracao do nosso datalakehouse local 

In [ ]:
# Executando Esse bloco de codigo criaremos as quatro tabelas no nosso banco de dados lakehouse

### TABELA REGISTRO DE ARQUIVO PROCESSADOS

con.execute(
    """CREATE TABLE IF NOT EXISTS processed_files(
        filename TEXT PRIMARY KEY,
        processed_at TIMESTAMP DEFAULT NOW()
    );"""
)

### TABELA REGISTRO DE MARCA DE AGUA ULTIMA ATUALIZACAO SILVER

con.execute(
    """CREATE TABLE IF NOT EXISTS silver_last_watermark(
        table_name TEXT PRIMARY KEY,
        last_year INT
    );"""
)

### TABELA REGISTRO DADOS PROCESSADOS DA GOLD

con.execute(
    """CREATE TABLE IF NOT EXISTS processed_gold_files(
        tableName TEXT,
        processed_at TIMESTAMP DEFAULT NOW()
    );"""
)

### TABELA REGISTRO DE LOGS CARGA DOS DADOS 

con.execute(
    """CREATE TABLE IF NOT EXISTS logsRegistros(
        table_name TEXT,
        status TEXT,
        date_register TIMESTAMP DEFAULT NOW()
    );"""
)

### TABELA REGISTRO DADOS DA SILVER 
#---------------------------------------------------------------------
# Essa tabela e a mais importante no nosso esquema datalakehouse local
# aqui nessa tabela que fica o coracao do sistema ela recebe os dados 
# brutos vindos do csv da bronze aplica pequenas transformacoes nesses 
# dados e armazena. 
# Construimos uma arquitetura manual nessa tabela para manter a regidez
# dos tipos de dados nao dexando para ser criado por enferencia conforme 
# fizemos com a tabela na camada bronze.
#---------------------------------------------------------------------

schema_sql = """
CREATE TABLE IF NOT EXISTS silver_clean (
    ID INTEGER,
    Name TEXT,
    Sex TEXT,
    Age INTEGER,
    Height FLOAT,
    Weight FLOAT,
    Team TEXT,
    NOC TEXT,
    Games TEXT,
    Year INTEGER,
    Season TEXT,
    City TEXT,
    Sport TEXT,
    Event TEXT,
    Medal TEXT
);
"""

con.execute(schema_sql)

##### Procesamento dos dados Para camada Bronze

In [ ]:
# ---------------------------------------------------------------------------------------------------
# PROCESSAR NOVOS CSVs → BRONZE
# Aqui os dados serao extraidos do ficheiro csv e passado para banco de dados duckdb lakehouse
# ---------------------------------------------------------------------------------------------------

# Leitura de todos Arquivos que estao na pasta bronze do formato csv
# certfiquemos aqui que nessas pasta so vai ter arquivos com mesma estrutura de dados
# Tambem para melhor gestao os arquivos nessas pasta serao salvos como exemplo dados_25/11/2025
# Podendo assim ser processado um arquivo por vez e arquivos processados nao serao mas processados

csv_files = glob(os.path.join(bronze_path, "*.csv"))

# cria uma lista com arquivos processados
new_files = []

# vai verificar de todos os arquivos da pasta bronze os que ja foram processados e armazenados na tabelas de
# arquivos processados, faz essas verificacao pelo nome por isso importante salvar nome e dados na tabela bronze
# Se o arquivo nao existir na tabela de arquivos processados sera adicionado a lista de new_files.

for f in csv_files:
    fname = os.path.basename(f)
    exists = con.execute("SELECT 1 FROM processed_files WHERE filename = ?", (fname,)).fetchone()
    if not exists:
        new_files.append(f)

# Aqui verfiquei se existe alguma arquivo para ser processado caso nao encontrar nenhum arquivo 
# retorna sem arquvivos serem processados caso tiver arquivos mostra esses arquivos que vao ser 
# processados e enviados para camada tabela bronze.

if not new_files:
    print("Nenhum novo arquivo CSV encontrado.")
else:
    print(f"Novos arquivos encontrados: {new_files}")

# Aqui que vai acontecer a passagem dos dados do ficheiro CSV para tabela Staging_row (Dados brutos)
# diferentes das outras tabelas essa tabela foi criada como copia fiel dos ficheiro CSV assim e comportamento
# dos Stanging_row sao copias de dados brutos na camada lakehouse bronze

# NOTA: nesse projecto admitimos que temos o ficheiro com mesma estrutura em unico csv , tivessemos tabelas de
# clientes, produtos , categorias etc adptariamos o staging_row para {'staging_row' + file} criaria uma tabela 
# row para cada um desses ficheiros.

for file in new_files:
    print(f"Processando: {file}")

    # Carregar CSV em staging temporária
    con.execute("DROP TABLE IF EXISTS staging_raw;")
    con.execute(f"""
        CREATE TEMP TABLE staging_raw AS
        SELECT *
        FROM read_csv_auto('{file}', NULLSTRING='', SAMPLE_SIZE=50000);
    """)

# SAMPLE_SIZE=50000 faz inferencia de tipo de dados com base nas primeiras 50000 linhas
# NULLSTRING dados nulos substitui por vazios 

    # Registrar arquivo como processado na tabela
    fname = os.path.basename(file)
    con.execute("INSERT INTO processed_files (filename) VALUES (?)", (fname,))

    # adiconar ao tabela de logs o status da carga

#### Processamento dos dados para camada Silver 

Coração do nosso Datalakehouse na silver já teremos dados prontos para trabalhar com Power BI , excel etc porque os dados estao limpos mas usaremos a silver como camada de transformação e pureza dos dados. Para consumo prepararemos a camada gold e views de consultas mais frequentes e modelagem estrela que ira ser consumida no Power BI

In [ ]:
# Descobrir maior 'Year' já carregado na Silver

res = con.execute("""
        SELECT last_year FROM silver_last_watermark
        WHERE table_name='silver_clean';
    """).fetchone()

# res[0] resultado da query em duckdb e uma lista aqui peguei o valor indice 0
# se esse res for TRUE retorna res[0] ou seja ultimo ano registro , caso contrario retorna None

last_year = res[0] if res else None

# Extrair linhas novas (incremental real)
# se last_year is none e porque nao tem nenhum dados na tabela
# vamos inserir dados na tabela silver_clean com dados ja carregados no staging_row

if last_year is None:
        print("Nenhum watermark encontrado — carga completa inicial da Silver.")
        con.execute("""
            INSERT INTO silver_clean
            SELECT
                ID::INT,
                TRIM(UPPER(Name)),
                Sex,
                Age::INT,
                Height::FLOAT,
                Weight::FLOAT,
                Team,
                NOC,
                Games,
                Year::INT,
                Season,
                City,
                Sport,
                Event,
                Medal
            FROM staging_raw;
        """)
# caso last_year tiver dados exemplo 2004 vai carregar dados da staging_row maiores que 2004
# caso no staging row nao foi passado arquivo dados > 2004 a query retorna nada nao carregando nada
# mantendo dados na silver ate 2004

else:
        print(f"Carregando apenas linhas novas com Year > {last_year}")
        con.execute(f"""
            INSERT INTO silver_clean
            SELECT
                ID::INT,
                TRIM(UPPER(Name)),
                Sex,
                Age::INT,
                Height::FLOAT,
                Weight::FLOAT,
                Team,
                NOC,
                Games,
                Year::INT,
                Season,
                City,
                Sport,
                Event,
                Medal
            FROM staging_raw
            WHERE Year::INT > {last_year};
        """)

# Atualizar marca de agua , assim mantendo a nossa atualizacao incremental com base em anos 
# mas podemos trabalhar isso para mes, dia ou ate id dependendo do caso etc

new_max_year = con.execute("SELECT MAX(Year) FROM silver_clean").fetchone()[0]
con.execute("""
        INSERT OR REPLACE INTO silver_last_watermark (table_name, last_year)
        VALUES ('silver_clean', ?)
    """, (new_max_year,))


#### Processamento dos dados para camada Gold

In [ ]:
# ---------------------------------------------------------------
# SILVER → GOLD (PARQUET POR ANO)
# Para nao ter carga de todos os anos a cada carga vamos atualizar 
# dados apenas de ultimos dois ano caso este tenham alteracoes serao
# atualizados , funcionaria mesmo para dias , meses e semanas
# ---------------------------------------------------------------

os.makedirs(gold_path, exist_ok=True)

# Etapa de verificacao de exitencia de dados na pasta gold

# se não existir nenhum diretorio dentro da pasta gol 
# segnifica que nenhuma carga foi feita antes realizara a primeira carga

if len(os.listdir(gold_path)) == 0:

    print("preparando a primeira carga dos dados...")

    anos_silver = con.execute("SELECT DISTINCT Year FROM silver_clean ORDER BY Year").fetchall()
    anos_silver = [a[0] for a in anos_silver]

    for ano in anos_silver:
        ano_path = os.path.join(gold_path, f"Year={ano}")
        os.makedirs(ano_path, exist_ok=True)
        parquet_file = os.path.join(ano_path, "part.parquet")

        # Escrever/reescrever apenas partições afetadas
        con.execute(f"""
            COPY (
                SELECT * FROM silver_clean WHERE Year = {ano}
            )
            TO '{parquet_file}'
            (FORMAT PARQUET);
        """)
else:
    anos_silver = con.execute("SELECT MAX(Year) FROM silver_clean ORDER BY Year").fetchall()

    # vai percorrer apenas ultimos dois anos de silver para rescrever
    anos_silver = [anos_silver[0] - 1 , anos_silver[0]]

    for ano in anos_silver:
        ano_path = os.path.join(gold_path, f"Year={ano}")
        os.makedirs(ano_path, exist_ok=True)
        parquet_file = os.path.join(ano_path, "part.parquet")

        # Escrever/reescrever apenas partições afetadas
        con.execute(f"""
            COPY (
                SELECT * FROM silver_clean WHERE Year = {ano}
            )
            TO '{parquet_file}'
            (FORMAT PARQUET);
        """) 

# regista na tabela processamento de ficheiro ouro as data da atualizacao desses dados
con.execute("INSERT INTO processed_gold_files (filename) VALUES ('gold_files')")

#### Criacao de View e modelagem de dados para consumo em Excel e Power BI

In [ ]:

# ===========================================
# MODELAGEM DE DADOS ESTRELA PARA POWER BI
# ===========================================

# criamos um banco de dados que vai conter as tabelas e os modelos
db_modelo = "duckdb/modeloDados.duckdb"

#conect  esse banco de dados
con = duckdb.connect(db_modelo)

# criando a dimensao atleta
con.execute("""
    CREATE TABLE DimAthlete AS
    SELECT DISTINCT
        ROW_NUMBER() OVER () AS athlete_key,
        ID AS athlete_id,
        Name,
        Sex,
        NOC,
        Height,
        Weight
    FROM silver_clean;
""")

# criando a dimensao Games
con.execute("""
    CREATE TABLE DimGames AS
    SELECT DISTINCT
        ROW_NUMBER() OVER () AS games_key,
        Games,
        Year,
        Season,
        City
    FROM silver_clean;
""")

# criando a facto de olimpiadas

# criando a dimensao Games
con.execute("""
    CREATE TABLE FactParticipation AS
    SELECT
        a.athlete_key,
        g.games_key,
        sc.Medal,
        sc.Age,
        sc.Height,
        sc.Weight
    FROM silver_clean sc
    LEFT JOIN DimAthlete a ON sc.ID = a.athlete_id
    LEFT JOIN DimGames g    ON sc.Games = g.Games;
""")

In [ ]:
# ====================================
# VIEW GOLD PARA POWER BI ou EXCEL
# ====================================
con.execute("""
CREATE OR REPLACE VIEW gold_athletes AS
SELECT * FROM parquet_scan('data_lake/gold/*/*.parquet');
""")

print("Pipeline Bronze → Silver → Gold concluído com sucesso!")